In [2]:
!pip install mlflow dagshub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Setup and MLflow Initialization
We connect to DagsHub and set the experiment to **GradientBoosting_Training**. Every run in this notebook is grouped under that experiment so results are clearly separated from other architectures on DagsHub.

In [3]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

# Connect to DagsHub
os.environ['MLFLOW_TRACKING_USERNAME'] = 'GigiSichinava'
dagshub.init(repo_owner='GigiSichinava', repo_name='ML-Assignment-2')
mlflow.set_tracking_uri("https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow")
mlflow.set_experiment("GradientBoosting_Training")

print("DagsHub connection established.")
print("Experiment: GradientBoosting_Training")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=25c215e2-82c4-441b-a945-e6199d052f24&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=6d30b67cea97389461a9bb7fa27b1191d9a22bb34ec6f1c2ff87d277eb4f8a78




Accessing as GigiSichinava

Initialized MLflow to track repo "GigiSichinava/ML-Assignment-2"

Repository GigiSichinava/ML-Assignment-2 initialized!

DagsHub connection established.
Experiment: GradientBoosting_Training


## Data Loading
We load both the transaction and identity tables then merge them on TransactionID with a left join. This keeps all transactions even those with no identity record.

In [4]:
# Load and merge the two tables on TransactionID
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity     = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')

# Left join: keeps all transactions even those without an identity record
train_df = train_transaction.merge(train_identity, on='TransactionID', how='left')
test_df  = test_transaction.merge(test_identity,  on='TransactionID', how='left')

print(f"Train shape: {train_df.shape}")
print(f"Test shape:  {test_df.shape}")
print(f"Fraud rate:  {train_df['isFraud'].mean():.4f}  ({train_df['isFraud'].sum()} fraud cases)")


Train shape: (590540, 434)
Test shape:  (506691, 433)
Fraud rate:  0.0350  (20663 fraud cases)


## Data Cleaning
Three cleaning steps are applied and logged as a single MLflow run. First we drop columns where more than 50% of values are missing. Then numeric NaNs are filled with the column median — robust to the heavy skew in financial data. Categorical NaNs are filled with the column mode.

In [5]:
with mlflow.start_run(run_name="GradientBoosting_Cleaning"):
    mlflow.log_param("stage", "Cleaning")

    # Separate target and drop the ID column
    y = train_df['isFraud'].copy()
    train_df.drop(columns=['isFraud', 'TransactionID'], inplace=True, errors='ignore')
    test_df.drop(columns=['TransactionID'], inplace=True, errors='ignore')

    # Step 1: Drop columns where more than 50% of values are missing
    # These columns carry too little information to be useful after imputation
    missing_ratio = train_df.isnull().mean()
    high_missing  = missing_ratio[missing_ratio > 0.5].index.tolist()
    train_df.drop(columns=high_missing, inplace=True)
    test_df.drop(columns=[c for c in high_missing if c in test_df.columns], inplace=True)
    print(f"Dropped {len(high_missing)} high-missing columns")

    # Identify numeric and categorical columns after dropping
    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()

    # Step 2: Fill numeric NaN with median
    # Median is more robust than mean because transaction amounts are heavily skewed
    for col in num_cols:
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        test_df[col]  = test_df[col].fillna(median_val)

    # Step 3: Fill categorical NaN with mode
    # The most frequent category is the safest default for unknown values
    for col in cat_cols:
        mode_val = train_df[col].mode()[0]
        train_df[col] = train_df[col].fillna(mode_val)
        test_df[col]  = test_df[col].fillna(mode_val)

    mlflow.log_param("dropped_columns",   len(high_missing))
    mlflow.log_param("numeric_features",  len(num_cols))
    mlflow.log_param("categorical_features", len(cat_cols))
    print(f"After cleaning: Train={train_df.shape}, Test={test_df.shape}")
    print("Cleaning run logged to MLflow.")


Dropped 214 high-missing columns
After cleaning: Train=(590540, 218), Test=(506691, 256)
Cleaning run logged to MLflow.
🏃 View run GradientBoosting_Cleaning at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5/runs/c74db88a17654e2ea2f6d5d3e5975fa9
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5


## Feature Engineering
We create five new features that capture fraud signals not visible in the raw columns. Log-transformed amount corrects for skew. The decimal part captures suspicious cent patterns. Hour and day features expose temporal fraud behaviour. All categorical columns are label-encoded since tree models handle integer inputs well.

In [6]:
# Feature 1: Log transform of transaction amount
# Transaction amounts are right skewed so log1p brings the distribution closer to normal
train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
test_df['TransactionAmt_log']  = np.log1p(test_df['TransactionAmt'])

# Feature 2: Decimal part of transaction amount
# Fraudsters often use amounts like 99.00 or 0.01 — the cents part captures this
train_df['TransactionAmt_decimal'] = train_df['TransactionAmt'] - train_df['TransactionAmt'].astype(int)
test_df['TransactionAmt_decimal']  = test_df['TransactionAmt'] - test_df['TransactionAmt'].astype(int)

# Feature 3: Hour of day extracted from TransactionDT timestamp
# Fraud tends to happen at unusual hours like late night
train_df['Transaction_hour'] = (train_df['TransactionDT'] // 3600) % 24
test_df['Transaction_hour']  = (test_df['TransactionDT']  // 3600) % 24

# Feature 4: Day of week extracted from TransactionDT
# Fraud patterns often differ by day of week
train_df['Transaction_day'] = (train_df['TransactionDT'] // (3600 * 24)) % 7
test_df['Transaction_day']  = (test_df['TransactionDT']  // (3600 * 24)) % 7

# Feature 5: Label encode all categorical columns
# Tree models handle integer encoded categories well without one-hot expansion
le = LabelEncoder()
for col in cat_cols:
    if col in train_df.columns:
        combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
        le.fit(combined)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col]  = le.transform(test_df[col].astype(str))

print(f"Feature engineering complete. Final shape: {train_df.shape}")


/tmp/ipykernel_57/1320925072.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df['TransactionAmt_log'] = np.log1p(train_df['TransactionAmt'])
/tmp/ipykernel_57/1320925072.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_df['TransactionAmt_log']  = np.log1p(test_df['TransactionAmt'])
/tmp/ipykernel_57/1320925072.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conca

Feature engineering complete. Final shape: (590540, 222)


## Feature Selection
Three sequential methods are applied and logged to MLflow. Variance threshold removes near-constant columns. Correlation filter removes redundant pairs above 0.95. Random Forest importance then selects the top 50 most predictive features.

In [7]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="GradientBoosting_Feature_Selection"):
    mlflow.log_param("stage", "Feature_Selection")

    X         = train_df.copy()
    X_test_fs = test_df.copy()

    # Method 1: Variance threshold removes near-constant columns
    # A feature with almost no variance cannot help the model distinguish fraud
    vt           = VarianceThreshold(threshold=0.01)
    vt.fit(X)
    low_var_cols = X.columns[~vt.get_support()].tolist()
    X            = X.drop(columns=low_var_cols)
    X_test_fs    = X_test_fs.drop(columns=[c for c in low_var_cols if c in X_test_fs.columns])
    print(f"After variance threshold: {X.shape[1]} features")

    # Method 2: Correlation filter removes redundant feature pairs above 0.95
    # Highly correlated features provide duplicate information and can slow training
    corr_matrix    = X.corr().abs()
    upper          = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    high_corr_cols = [col for col in upper.columns if any(upper[col] > 0.95)]
    X              = X.drop(columns=high_corr_cols)
    X_test_fs      = X_test_fs.drop(columns=[c for c in high_corr_cols if c in X_test_fs.columns])
    print(f"After correlation filter: {X.shape[1]} features")

    # Method 3: Random Forest importance keeps the top 50 most predictive features
    rf_selector  = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
    rf_selector.fit(X, y)
    importances  = pd.Series(rf_selector.feature_importances_, index=X.columns)
    top_features = importances.nlargest(50).index.tolist()
    X_selected      = X[top_features]
    X_test_selected = X_test_fs[top_features]
    print(f"Top 50 features selected by Random Forest importance")

    mlflow.log_param("features_after_variance", X.shape[1])
    mlflow.log_param("features_after_corr",     X.shape[1] - len(high_corr_cols))
    mlflow.log_param("final_feature_count",      len(top_features))

    # Stratified split: preserves the fraud ratio in both train and validation sets
    X_train, X_val, y_train, y_val = train_test_split(
        X_selected, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Train: {X_train.shape}, Val: {X_val.shape}")
    print("Feature selection run logged to MLflow.")


After variance threshold: 199 features
After correlation filter: 144 features
Top 50 features selected by Random Forest importance
Train: (472432, 50), Val: (118108, 50)
Feature selection run logged to MLflow.
🏃 View run GradientBoosting_Feature_Selection at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5/runs/bc8c38ecf6e1448dadf4a371dc32fa47
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5


## Model Training
We test multiple Gradient Boosting configurations by varying n_estimators and learning rate. Each run is logged to MLflow to compare performance across configurations.

In [8]:
# Utility function: trains the model, logs all metrics to MLflow and saves to the registry
def train_and_log(model, model_name):
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        train_preds = model.predict_proba(X_train)[:, 1]
        val_preds   = model.predict_proba(X_val)[:, 1]

        train_auc = roc_auc_score(y_train, train_preds)
        val_auc   = roc_auc_score(y_val,   val_preds)
        gap       = train_auc - val_auc

        mlflow.log_param("model_type",  model_name)
        mlflow.log_metric("train_auc",  train_auc)
        mlflow.log_metric("val_auc",    val_auc)
        mlflow.log_metric("overfit_gap", gap)

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            registered_model_name=model_name
        )

        print(f"Model logged: {model_name}")
        print(f"   Train AUC:   {train_auc:.4f}")
        print(f"   Val AUC:     {val_auc:.4f}")
        print(f"   Overfit gap: {gap:.4f}")
        print("-" * 30)

        return model


In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# n_estimators sweep:
gbr_options = [50, 100]
print("=== n_estimators Sweep ===")
for n in gbr_options:
    label = f"GBR_n_estimators_{n}"
    model = GradientBoostingClassifier(n_estimators=n, random_state=42)
    with mlflow.start_run(run_name=label):
        cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
        model.fit(X_train, y_train)
        train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
        val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])
        gap       = train_auc - val_auc
        mlflow.log_param("model_type",  label)
        mlflow.log_param("n_estimators", n)
        mlflow.log_metric("train_auc",   train_auc)
        mlflow.log_metric("val_auc",     val_auc)
        mlflow.log_metric("cv_auc_mean", cv_scores.mean())
        mlflow.log_metric("cv_auc_std",  cv_scores.std())
        mlflow.log_metric("overfit_gap", gap)
        results.append({"label": label, "train_auc": train_auc, "val_auc": val_auc, "gap": gap})
        print(f"{label}: Train={train_auc:.4f} Val={val_auc:.4f} Gap={gap:.4f}")
        if n <= 50:
            print("   Note: Few estimators — likely underfitting.")
    print("-" * 30)

print("\nGradient Boosting training complete.")


=== n_estimators Sweep ===
GBR_n_estimators_50: Train=0.8593 Val=0.8566 Gap=0.0027
   Note: Few estimators — likely underfitting.
🏃 View run GBR_n_estimators_50 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5/runs/5debfb333cfa4c1dba79098c0a5862b8
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5
------------------------------
GBR_n_estimators_100: Train=0.8674 Val=0.8643 Gap=0.0031
🏃 View run GBR_n_estimators_100 at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5/runs/a656233e252440f99a09d67b5bccae61
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5
------------------------------

Gradient Boosting training complete.


## Best Pipeline Registration
The best GradientBoosting configuration is wrapped in a sklearn Pipeline together with a preprocessing function. This pipeline can run directly on raw unprocessed data because it applies all cleaning and feature engineering steps internally. The pipeline is saved to the DagsHub Model Registry.

In [14]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

# This function applies the exact same steps as the cleaning and feature
# engineering cells above, so the pipeline can run on raw unprocessed data
def preprocess_raw(df):
    df = df.copy()

    # Preprocessing step: drop high missing columns using the same 50% threshold
    missing_ratio = df.isnull().mean()
    df = df.drop(columns=missing_ratio[missing_ratio > 0.5].index.tolist(), errors='ignore')

    num_c = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_c = df.select_dtypes(include=['object']).columns.tolist()

    # Fill missing values: same strategy as in the Cleaning section
    for col in num_c:
        df[col] = df[col].fillna(df[col].median())
    for col in cat_c:
        df[col] = df[col].fillna(df[col].mode()[0] if len(df[col].mode()) > 0 else 'Unknown')

    # Apply engineered features: same ones created in the Feature Engineering section
    df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
    df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
    df['Transaction_hour']       = (df['TransactionDT'] // 3600) % 24
    df['Transaction_day']        = (df['TransactionDT'] // (3600 * 24)) % 7

    # Label encode categoricals
    le = LabelEncoder()
    for col in cat_c:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))

    # Keep only top features: the ones selected during Feature Selection
    keep = [c for c in top_features if c in df.columns]
    return df[keep]

from sklearn.ensemble import GradientBoostingClassifier
best_gbr = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=4,
    subsample=0.8, random_state=42
)

# Build pipeline: combine preprocessing and the classifier into one object
# This pipeline runs directly on raw unprocessed data
gradientboosting_pipeline = Pipeline([
    ('preprocessor', FunctionTransformer(preprocess_raw)),
    ('classifier',   best_gbr)
])

# Fit on raw data: the pipeline handles all preprocessing internally
train_raw = train_transaction.merge(train_identity, on='TransactionID', how='left')
y_raw     = train_raw['isFraud'].copy()
train_raw = train_raw.drop(columns=['isFraud', 'TransactionID'], errors='ignore')

gradientboosting_pipeline.fit(train_raw, y_raw)
print("GradientBoosting Pipeline trained on raw data successfully.")

# Register to Model Registry: save the pipeline for later use in inference
with mlflow.start_run(run_name="GradientBoosting_Pipeline_Registration"):
    mlflow.log_param("model_type", "GradientBoosting_Pipeline")
    mlflow.sklearn.log_model(
        sk_model=gradientboosting_pipeline,
        artifact_path="model",
        registered_model_name="GradientBoosting_Pipeline"
    )
    print("GradientBoosting Pipeline registered in DagsHub Model Registry.")

/tmp/ipykernel_57/115795294.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
/tmp/ipykernel_57/115795294.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)
/tmp/ipykernel_57/115795294.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

GradientBoosting Pipeline trained on raw data successfully.


2026/05/04 01:09:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 01:09:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'GradientBoosting_Pipeline' already exists. Creating a new version of this model...
2026/05/04 01:10:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: GradientBoosting_Pipeline, version 2
Created version '2' of model 'GradientBoosting_Pipeline'.


GradientBoosting Pipeline registered in DagsHub Model Registry.
🏃 View run GradientBoosting_Pipeline_Registration at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5/runs/78547edce21e4f01839d554430f2eb9a
🧪 View experiment at: https://dagshub.com/GigiSichinava/ML-Assignment-2.mlflow/#/experiments/5
